In [1]:
import math
import copy 

import torch
import datasets
import torchvision
import tensordict
import tqdm.auto
import plotly.express
import pandas as pd
from lovely_tensors.repr_str import lovely

from IPython.display import clear_output


- 0-1 instead of mask (smooth mask)
- Remove top-agreements as well as the top disagreements
- Different rules of computing the control update (instead of average)
- remove less than 1% of disagreements
- Neuron update - row wise instead of parameter wise
- Different control batches should agree on removed gradients - measure that somehow
- looke where in the model is the removed gradients
- Disagreement threshold / remove 10% of total sum disagreement rather than quantile
- I'm also using adamw instead of SGD, so I might have some confounding factors I'm not aware of

In [2]:
device = "cuda:7"
dtype = torch.float32

In [3]:
def preprocess(example: dict):
    return {
        "x": example["x"] / 255.0,
    }

def load_mnist():
    data = (
        datasets.load_dataset("ylecun/mnist")
        .rename_columns({"image": "x", "label": "y"})
    )
    data.set_format("torch")
    data = data.map(preprocess)
    return data

def load_cifar100():
    data = (
        datasets.load_dataset("uoft-cs/cifar100")
        .rename_columns({"img": "x", "fine_label": "y"})
    )
    data.set_format("torch")
    data = data.map(preprocess)
    return data

data = load_cifar100()

In [4]:
train_data = tensordict.TensorDict(
    {
        "x": data["train"]["x"][:].to(device, dtype),
        "y": data["train"]["y"][:].to(device)
    },
    device=device,
    batch_size=len(data["train"])
)

valid_data = tensordict.TensorDict(
    {
        "x": data["test"]["x"][:].to(device, dtype),
        "y": data["test"]["y"][:].to(device)
    },
    device=device,
    batch_size=len(data["test"])
)

train_eval_idcs = torch.randperm(len(train_data), device=device, generator=torch.Generator(device).manual_seed(0))[:10000]
train_eval_idcs

lovely(train_data["x"])

tensor[50000, 3, 32, 32] n=153600000 (0.6Gb) x∈[0., 1.000] μ=0.478 σ=0.268 cuda:7

In [5]:
example = train_data[0]
print(lovely(example["x"]))
print("label: ", example["y"])

tensor[3, 32, 32] n=3072 (12Kb) x∈[0.004, 1.000] μ=0.542 σ=0.284 cuda:7
label:  tensor(19, device='cuda:7')


In [6]:
n_classes = max(train_data["y"]).item() + 1
print(f"Number of classes: {n_classes}")

Number of classes: 100


In [7]:
in_channels = train_data["x"][0].shape[0]
print(f"Number of input channels: {in_channels}")

Number of input channels: 3


In [8]:
batch_size = 256

In [9]:
def make_efficientnet(n_classes: int, in_channels: int, device: torch.device, dtype: torch.dtype) -> torch.nn.Module:
    model = torchvision.models.efficientnet_b0(num_classes=n_classes).to(device, dtype=dtype)
    if in_channels != 3:
        in_conv = model.features[0][0]
        model.features[0][0] = torch.nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_conv.out_channels,
            kernel_size=in_conv.kernel_size,
            stride=in_conv.stride,
            padding=in_conv.padding,
            bias=in_conv.bias is not None,
        ).to(device, dtype=dtype)
    return model

def make_resnet(n_classes: int, in_channels: int, device: torch.device, dtype: torch.dtype, weights=None) -> torch.nn.Module:
    if weights is not None:
        model = torchvision.models.resnet18(weights=weights).to(device, dtype=dtype)
        model.fc = torch.nn.Linear(model.fc.in_features, n_classes).to(device, dtype=dtype)
    else:
        model = torchvision.models.resnet18(num_classes=n_classes).to(device, dtype=dtype)
    
    if in_channels != 3:
        in_conv = model.conv1
        model.conv1 = torch.nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_conv.out_channels,
            kernel_size=in_conv.kernel_size,
            stride=in_conv.stride,
            padding=in_conv.padding,
            bias=in_conv.bias is not None,
        ).to(device, dtype=dtype)
    
    return model

def get_acc(model: torch.nn.Module, data: tensordict.TensorDict) -> float:
    with torch.no_grad():
        model.eval()
        acc = model(data["x"]).argmax(dim=-1).eq(data["y"]).float().mean().item()
    return acc

def get_accuracies(model: torch.nn.Module, splits: dict[str, tensordict.TensorDict]) -> float:
    accs = {}
    for split, data in splits.items():
        accs[split] = get_acc(model, data)
    return accs

def plot_accuracies(x_axis: str, accs: list[dict], max_x_value: int) -> plotly.express.Figure:
    return plotly.express.line(pd.DataFrame(accs), x=x_axis, y=list(accs[0].keys()), title="Accuracy", range_y=[0, 1], range_x=[0, max_x_value])

In [10]:
MAX_STEPS = 10000
LOG_EVERY = 100
lr = 5e-5

In [42]:
torch.manual_seed(0)
model = make_resnet(n_classes=n_classes, in_channels=in_channels, device=device, dtype=dtype)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

accs_standard = []
for step in tqdm.auto.trange(MAX_STEPS+1):
    batch_start_idx = (step * batch_size) % len(train_data)
    batch = train_data[batch_start_idx:batch_start_idx + batch_size]
    logits = model(batch["x"])
    loss = torch.nn.functional.cross_entropy(logits, batch["y"])
    optim.zero_grad()
    loss.backward()
    optim.step()
    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_standard.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_standard, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_standard)

,step,train,valid
0,0,0.0116,0.0094
1,100,0.1146,0.0960
2,200,0.1749,0.1456
3,300,0.2228,0.1713
4,400,0.2608,0.1918
...,...,...,...
96,9600,0.9995,0.2319
97,9700,0.9998,0.2365
98,9800,0.9997,0.2326
99,9900,0.9990,0.2319


In [46]:
torch.manual_seed(0)
model = make_resnet(n_classes=n_classes, in_channels=in_channels, device=device, dtype=dtype, weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

accs_standard = []
for step in tqdm.auto.trange(MAX_STEPS+1):
    batch_start_idx = (step * batch_size) % len(train_data)
    batch = train_data[batch_start_idx:batch_start_idx + batch_size]
    logits = model(batch["x"])
    loss = torch.nn.functional.cross_entropy(logits, batch["y"])
    optim.zero_grad()
    loss.backward()
    optim.step()
    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_standard.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_standard, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_standard)

KeyboardInterrupt: 

In [ ]:
torch.manual_seed(0)
model = make_resnet(n_classes=n_classes, in_channels=in_channels, device=device)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

accs_standard = []
for step in tqdm.auto.trange(MAX_STEPS+1):
    batch_start_idx = (step * batch_size) % len(train_data)
    batch = train_data[batch_start_idx:batch_start_idx + batch_size]
    logits = model(batch["x"])
    loss = torch.nn.functional.cross_entropy(logits, batch["y"])
    optim.zero_grad()
    loss.backward()
    optim.step()
    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_standard.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_standard, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_standard)

In [ ]:
torch.manual_seed(0)
model = make_efficientnet(n_classes=n_classes, in_channels=in_channels, device=device)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

n_agg = 2

accs_agg_min2 = []

for step in tqdm.auto.trange(MAX_STEPS+1):
    grads = []
    orig_grad_norm = 0
    for inner_step in range(n_agg):
        optim.zero_grad()
        batch = train_data[torch.randint(0, len(train_data), (batch_size,), device=device)]
        logits = model(batch["x"].to(device))
        loss = torch.nn.functional.cross_entropy(logits, batch["y"].to(device))
        grads.append(torch.autograd.grad(loss, model.parameters(), create_graph=True))
        if inner_step == 0:
            orig_grad_norm = math.sqrt(sum(g.norm().item() ** 2 for g in grads[0]))
    
    # take minimum (closest to zero) gradient across the n_agg gradients for each parameter
    # rescale to get original grad norm
    for p, *gs in zip(model.parameters(), *grads):
        p_grads = torch.stack(gs, dim=0)
        abs_grads = p_grads.abs()
        min_values, min_idcs = abs_grads.min(dim=0)
        min_grad = min_values * p_grads.sign().gather(0, min_idcs.unsqueeze(0)).squeeze(0)
        p.grad = min_grad
    
    min_grad_norm = math.sqrt(sum(p.grad.norm().item() ** 2 for p in model.parameters()))
    scale_factor = orig_grad_norm / (min_grad_norm + 1e-8)
    for p in model.parameters():
        p.grad *= scale_factor

    optim.step()
    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_agg_min2.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_agg_min2, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_agg_min2)

In [ ]:
torch.manual_seed(0)
model = make_efficientnet(n_classes=n_classes, in_channels=in_channels, device=device)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

n_agg = 4

for i in tqdm.auto.tqdm(range(MAX_STEPS+1)):
    grads = []
    for _ in range(n_agg):
        optim.zero_grad()
        batch = train_data[torch.randint(0, len(train_data), (batch_size//n_agg,), device=device)]
        logits = model(batch["x"].to(device))
        loss = torch.nn.functional.cross_entropy(logits, batch["y"].to(device))
        grads.append(torch.autograd.grad(loss, model.parameters(), create_graph=True))
    
    # take minimum (closest to zero) gradient across the n_agg gradients for each parameter
    for p, *gs in zip(model.parameters(), *grads):
        p_grads = torch.stack(gs, dim=0)
        abs_grads = p_grads.abs()
        min_values, min_idcs = abs_grads.min(dim=0)
        min_grad = min_values * p_grads.sign().gather(0, min_idcs.unsqueeze(0)).squeeze(0)
        p.grad = min_grad

    optim.step()
    if i % LOG_EVERY == 0:
        acc = model(valid_data["x"]).argmax(dim=-1).eq(valid_data["y"]).float().mean().item()
        print(f"Iteration {i}: val acc = {acc:.4f}")

In [ ]:
torch.manual_seed(0)
model = make_efficientnet(n_classes=n_classes, in_channels=in_channels, device=device)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

n_agg = 2

for i in tqdm.auto.tqdm(range(MAX_STEPS+1)):
    grads = []
    for _ in range(n_agg):
        optim.zero_grad()
        batch = train_data[torch.randint(0, len(train_data), (batch_size//n_agg,), device=device)]
        logits = model(batch["x"].to(device))
        loss = torch.nn.functional.cross_entropy(logits, batch["y"].to(device))
        grads.append(torch.autograd.grad(loss, model.parameters(), create_graph=True))
    
    # take elementwise product (soft intersection) of gradient across the n_agg gradients for each parameter
    for p, *gs in zip(model.parameters(), *grads):
        p_grads = torch.stack(gs, dim=0)
        p.grad = p_grads.prod(dim=0)

    optim.step()
    if i % LOG_EVERY == 0:
        acc = model(valid_data["x"]).argmax(dim=-1).eq(valid_data["y"]).float().mean().item()
        print(f"Iteration {i}: val acc = {acc:.4f}")

In [ ]:
torch.manual_seed(0)
model = make_efficientnet(n_classes=n_classes, in_channels=in_channels, device=device)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

aux_model = make_efficientnet(n_classes=n_classes, in_channels=in_channels, device=device)
for i in tqdm.auto.tqdm(range(MAX_STEPS+1)):
    batch = train_data[torch.randint(0, len(train_data), (batch_size,), device=device)]
    update = {k: torch.zeros_like(v) for k, v in model.named_parameters()}
    init_grad_norms = {}

    aux_model.load_state_dict({k: v for k, v in model.named_buffers()}, strict=False)
    for inner_steps in range(10):
        for param in aux_model.parameters():
            param.grad = None
        # merge the model with the update:
        aux_model.load_state_dict({k: v + update[k] for k, v in model.named_parameters()}, strict=False)
        logits = aux_model(batch["x"].to(device))
        loss = torch.nn.functional.cross_entropy(logits, batch["y"].to(device))
        loss.backward()
        if inner_steps == 0:
            for k, v in aux_model.named_parameters():
                if v.grad is not None:
                    init_grad_norms[k] = v.grad.norm().item()
        for k, v in aux_model.named_parameters():
            if v.grad is not None:
                update[k] += v.grad
                update[k] *= 0.99

    optim.zero_grad()
    for k, v in update.items():
        model.get_parameter(k).grad = v * (init_grad_norms[k] / v.norm())
    optim.step()

    if i % 10 == 0:
        acc = model(valid_data["x"]).argmax(dim=-1).eq(valid_data["y"]).float().mean().item()
        print(f"Iteration {i}: val acc = {acc:.4f}")

In [ ]:
def flatten_tensors(tensor_list):
    return torch.cat([t.view(-1) for t in tensor_list])

def get_global_norm(tensor_list):
    return torch.norm(flatten_tensors(tensor_list))

keep_top_p = 0.9        # Keep top p-fraction of gradients
control_mult = 4         # Control batch is 4x larger

torch.manual_seed(0)
model = make_resnet(n_classes=n_classes, in_channels=in_channels, device=device, dtype=dtype)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

accs_filter_disagreements = []

for step in tqdm.auto.trange(MAX_STEPS+1):
    # 1. Sample Batches
    batch_start_idx = (step * batch_size) % len(train_data)
    batch = train_data[batch_start_idx:batch_start_idx + batch_size]
    
    ctrl_idxs = torch.randint(0, len(train_data), (batch_size * control_mult,), device=device)
    control_batch = train_data[ctrl_idxs]

    # 2. Compute Gradients (separately)
    logits = model(batch["x"])
    loss = torch.nn.functional.cross_entropy(logits, batch["y"])
    train_grads = torch.autograd.grad(loss, model.parameters(), create_graph=False)

    ctrl_logits = model(control_batch["x"])
    ctrl_loss = torch.nn.functional.cross_entropy(ctrl_logits, control_batch["y"])
    ctrl_grads = torch.autograd.grad(ctrl_loss, model.parameters(), create_graph=False)

    # 3. Compute Alignment Scores & Threshold
    scores = [g_train * g_ctrl for g_train, g_ctrl in zip(train_grads, ctrl_grads)]
    flat_scores = flatten_tensors(scores)
    threshold = torch.quantile(flat_scores, 1 - keep_top_p)

    # 4. Filter Gradients (Store temporarily)
    filtered_grads = []
    for g_train, score in zip(train_grads, scores):
        mask = (score >= threshold).float()
        filtered_grads.append(g_train * mask)

    # Calculate the magnitude (norm) of the original raw gradients
    orig_norm = get_global_norm(train_grads)
    
    # Calculate the magnitude of the gradients after we masked 80% of them
    current_norm = get_global_norm(filtered_grads)
    
    # Calculate scaling factor (safe division)
    # If we kept 20% of grads, current_norm is likely much smaller than orig_norm.
    # We want to scale up so the update step size remains consistent.
    scale_factor = orig_norm / (current_norm + 1e-8)
    
    # 5. Assign to Model & Step
    optim.zero_grad()
    for param, f_grad in zip(model.parameters(), filtered_grads):
        # Apply the scale factor here
        param.grad = f_grad # * scale_factor #TODO
        
    optim.step()

    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_filter_disagreements.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_filter_disagreements, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_filter_disagreements)

,step,train,valid
0,0,0.0128,0.0114
1,100,0.1658,0.1232
2,200,0.2483,0.1692
3,300,0.3301,0.1955
4,400,0.4053,0.2051
...,...,...,...
96,9600,0.9999,0.2174
97,9700,0.9999,0.2191
98,9800,0.9999,0.2182
99,9900,0.9998,0.2192


In [12]:
def flatten_tensors(tensor_list):
    return torch.cat([t.view(-1) for t in tensor_list])

def get_global_norm(tensor_list):
    return torch.norm(flatten_tensors(tensor_list))

In [ ]:


keep_top_p = 0.99        # Keep top p-fraction of gradients
control_mult = 4         # Control batch is 4x larger

torch.manual_seed(0)
model = make_resnet(n_classes=n_classes, in_channels=in_channels, device=device, dtype=dtype)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

accs_filter_disagreements = []

for step in tqdm.auto.trange(MAX_STEPS+1):
    # 1. Sample Batches
    batch_start_idx = (step * batch_size) % len(train_data)
    batch = train_data[batch_start_idx:batch_start_idx + batch_size]
    
    ctrl_idxs = torch.randint(0, len(train_data), (batch_size * control_mult,), device=device)
    control_batch = train_data[ctrl_idxs]

    # 2. Compute Gradients (separately)
    logits = model(batch["x"])
    loss = torch.nn.functional.cross_entropy(logits, batch["y"])
    train_grads = torch.autograd.grad(loss, model.parameters(), create_graph=False)

    ctrl_logits = model(control_batch["x"])
    ctrl_loss = torch.nn.functional.cross_entropy(ctrl_logits, control_batch["y"])
    ctrl_grads = torch.autograd.grad(ctrl_loss, model.parameters(), create_graph=False)

    # 3. Compute Alignment Scores & Threshold
    scores = [g_train * g_ctrl for g_train, g_ctrl in zip(train_grads, ctrl_grads)]
    flat_scores = flatten_tensors(scores)
    threshold = torch.quantile(flat_scores, 1 - keep_top_p)

    # 4. Filter Gradients (Store temporarily)
    filtered_grads = []
    for g_train, score in zip(train_grads, scores):
        mask = (score >= threshold).float()
        filtered_grads.append(g_train * mask)

    # Calculate the magnitude (norm) of the original raw gradients
    orig_norm = get_global_norm(train_grads)
    
    # Calculate the magnitude of the gradients after we masked 80% of them
    current_norm = get_global_norm(filtered_grads)
    
    # Calculate scaling factor (safe division)
    # If we kept 20% of grads, current_norm is likely much smaller than orig_norm.
    # We want to scale up so the update step size remains consistent.
    scale_factor = orig_norm / (current_norm + 1e-8)
    
    # 5. Assign to Model & Step
    optim.zero_grad()
    for param, f_grad in zip(model.parameters(), filtered_grads):
        # Apply the scale factor here
        param.grad = f_grad * scale_factor
        
    optim.step()

    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_filter_disagreements.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_filter_disagreements, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_filter_disagreements)

KeyboardInterrupt: 

In [72]:
keep_top_p = 0.99        # Keep top p-fraction of gradients
control_mult = 4         # Control batch is 4x larger

torch.manual_seed(0)
model = make_resnet(n_classes=n_classes, in_channels=in_channels, device=device, dtype=dtype)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

accs_filter_disagreements = []

for step in tqdm.auto.trange(MAX_STEPS+1):
    # 1. Sample Batches
    batch_start_idx = (step * batch_size) % len(train_data)
    batch = train_data[batch_start_idx:batch_start_idx + batch_size]
    
    ctrl_idxs = torch.randint(0, len(valid_data), (batch_size * control_mult,), device=device)
    control_batch = valid_data[ctrl_idxs]

    # 2. Compute Gradients (separately)
    logits = model(batch["x"])
    loss = torch.nn.functional.cross_entropy(logits, batch["y"])
    train_grads = torch.autograd.grad(loss, model.parameters(), create_graph=False)

    ctrl_logits = model(control_batch["x"])
    ctrl_loss = torch.nn.functional.cross_entropy(ctrl_logits, control_batch["y"])
    ctrl_grads = torch.autograd.grad(ctrl_loss, model.parameters(), create_graph=False)

    # 3. Compute Alignment Scores & Threshold
    scores = [g_train * g_ctrl for g_train, g_ctrl in zip(train_grads, ctrl_grads)]
    flat_scores = flatten_tensors(scores)
    threshold = torch.quantile(flat_scores, 1 - keep_top_p)

    # 4. Filter Gradients (Store temporarily)
    filtered_grads = []
    for g_train, score in zip(train_grads, scores):
        mask = (score >= threshold).float()
        filtered_grads.append(g_train * mask)

    # Calculate the magnitude (norm) of the original raw gradients
    orig_norm = get_global_norm(train_grads)
    
    # Calculate the magnitude of the gradients after we masked 80% of them
    current_norm = get_global_norm(filtered_grads)
    
    # Calculate scaling factor (safe division)
    # If we kept 20% of grads, current_norm is likely much smaller than orig_norm.
    # We want to scale up so the update step size remains consistent.
    scale_factor = orig_norm / (current_norm + 1e-8)
    
    # 5. Assign to Model & Step
    optim.zero_grad()
    for param, f_grad in zip(model.parameters(), filtered_grads):
        # Apply the scale factor here
        param.grad = f_grad * scale_factor
        
    optim.step()

    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_filter_disagreements.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_filter_disagreements, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_filter_disagreements)

KeyboardInterrupt: 

In [13]:
keep_top_p = 0.01        # Keep top p-fraction of gradients
control_mult = 4         # Control batch is 4x larger

torch.manual_seed(0)
model = make_resnet(n_classes=n_classes, in_channels=in_channels, device=device, dtype=dtype)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

accs_filter_disagreements = []

for step in tqdm.auto.trange(MAX_STEPS+1):
    # 1. Sample Batches
    batch_start_idx = (step * batch_size) % len(train_data)
    batch = train_data[batch_start_idx:batch_start_idx + batch_size]
    
    ctrl_idxs = torch.randint(0, len(valid_data), (batch_size * control_mult,), device=device)
    control_batch = valid_data[ctrl_idxs]

    # 2. Compute Gradients (separately)
    logits = model(batch["x"])
    loss = torch.nn.functional.cross_entropy(logits, batch["y"])
    train_grads = torch.autograd.grad(loss, model.parameters(), create_graph=False)

    ctrl_logits = model(control_batch["x"])
    ctrl_loss = torch.nn.functional.cross_entropy(ctrl_logits, control_batch["y"])
    ctrl_grads = torch.autograd.grad(ctrl_loss, model.parameters(), create_graph=False)

    # 3. Compute Alignment Scores & Threshold
    scores = [g_train * g_ctrl for g_train, g_ctrl in zip(train_grads, ctrl_grads)]
    flat_scores = flatten_tensors(scores)
    threshold = torch.quantile(flat_scores, 1 - keep_top_p)

    # 4. Filter Gradients (Store temporarily)
    filtered_grads = []
    for g_train, score in zip(train_grads, scores):
        mask = (score >= threshold).float()
        filtered_grads.append(g_train * mask)

    # Calculate the magnitude (norm) of the original raw gradients
    orig_norm = get_global_norm(train_grads)
    
    # Calculate the magnitude of the gradients after we masked 80% of them
    current_norm = get_global_norm(filtered_grads)
    
    # Calculate scaling factor (safe division)
    # If we kept 20% of grads, current_norm is likely much smaller than orig_norm.
    # We want to scale up so the update step size remains consistent.
    scale_factor = orig_norm / (current_norm + 1e-8)
    
    # 5. Assign to Model & Step
    optim.zero_grad()
    for param, f_grad in zip(model.parameters(), filtered_grads):
        # Apply the scale factor here
        param.grad = f_grad * scale_factor
        
    optim.step()

    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_filter_disagreements.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_filter_disagreements, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_filter_disagreements)

,step,train,valid
0,0,0.0124,0.0111
1,100,0.0483,0.1067
2,200,0.0941,0.2402
3,300,0.1238,0.3522
4,400,0.1456,0.4629
...,...,...,...
96,9600,0.6694,0.9998
97,9700,0.6741,0.9998
98,9800,0.6874,0.9998
99,9900,0.6888,0.9998


In [67]:
torch.manual_seed(0)
model = make_resnet(n_classes=n_classes, in_channels=in_channels, device=device, dtype=dtype)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

accs_subtract_overfitting = []

for step in tqdm.auto.trange(MAX_STEPS+1):
    batch_start_idx = (step * batch_size) % len(train_data)
    batch = train_data[batch_start_idx:batch_start_idx + batch_size]

    logits = model(batch["x"])
    loss = torch.nn.functional.cross_entropy(logits, batch["y"])
    optim.zero_grad()
    loss.backward()

    # perform 10 steps of SGD on the batch to get "overfitting update"
    # L1-regularize the *difference* between the original and control model's parameters to encourage sparsity in the overfitting update
    ctrl_model = copy.deepcopy(model)
    for _ in range(10):
        ctrl_logits = ctrl_model(batch["x"])
        ctrl_optim = torch.optim.SGD(ctrl_model.parameters(), lr=lr)
        ctrl_loss = torch.nn.functional.cross_entropy(ctrl_logits, batch["y"])
        ctrl_optim.zero_grad()
        ctrl_loss.backward()
        ctrl_optim.step()
        with torch.no_grad():
            for orig, ctrl in zip(model.parameters(), ctrl_model.parameters()):
                ctrl[:] = ctrl - 0.0001 * torch.sign(ctrl - orig)
            
    overfit_update = [ctrl_p - p for ctrl_p, p in zip(ctrl_model.parameters(), model.parameters())]
    print("sparsity of overfitting update:", (flatten_tensors(overfit_update).abs() < 0.000001).float().mean().item())

    # subtract the update from the current model's gradients
    with torch.no_grad():
        for p, u in zip(model.parameters(), overfit_update):
            p.grad -= u

    optim.step()

    if step % LOG_EVERY == 0:
        curr_accs = get_accuracies(model, {"train": train_data[train_eval_idcs], "valid": valid_data})
        accs_subtract_overfitting.append({"step": step, **curr_accs})
        clear_output(wait=True)
        display(plot_accuracies(x_axis="step", accs=accs_subtract_overfitting, max_x_value=MAX_STEPS))
        model.train()

pd.DataFrame(accs_subtract_overfitting)

,step,train,valid
0,0,0.0116,0.0094
1,100,0.1151,0.0945
2,200,0.1748,0.1448
3,300,0.2207,0.1702
4,400,0.2632,0.1900
...,...,...,...
96,9600,0.9998,0.2355
97,9700,0.9998,0.2358
98,9800,0.9998,0.2338
99,9900,0.9998,0.2357


In [ ]:
torch.manual_seed(0)
model = make_efficientnet(n_classes=n_classes, in_channels=in_channels, device=device)
optim = torch.optim.AdamW(model.parameters(), lr=lr)

for i in tqdm.auto.tqdm(range(MAX_STEPS+1)):
    batch = train_data[torch.randint(0, len(train_data), (batch_size*5,), device=device)]
    logits = model(batch["x"].to(device))
    loss = torch.nn.functional.cross_entropy(logits, batch["y"].to(device))
    optim.zero_grad()
    loss.backward()
    optim.step()
    if i % LOG_EVERY == 0:
        acc = model(valid_data["x"]).argmax(dim=-1).eq(valid_data["y"]).float().mean().item()
        print(f"Iteration {i}: val acc = {acc:.4f}")

In [ ]:
from torch.func import functional_call

# --- Configuration ---
keep_top_p = 0.20           # Quantile threshold (Top 20%)
mask_lr = 0.1               # Learning rate for the mask optimization
mask_reg = 0.05             # Regularization strength (pull mask to 0)
mask_steps = 5              # How many optimization steps for the mask
control_mult = 4            # Control batch size multiplier
model_lr = lr             # Must match your main optimizer's LR
# ---------------------

torch.manual_seed(0)
model = make_efficientnet(n_classes=n_classes, in_channels=in_channels, device=device)
optim = torch.optim.AdamW(model.parameters(), lr=model_lr)

# We need a copy of the model structure for the functional call
# (It doesn't need to hold weights, just the architecture)
aux_model = make_efficientnet(n_classes=n_classes, in_channels=in_channels, device=device)

def flatten_tensors(tensor_list):
    return torch.cat([t.view(-1) for t in tensor_list])

for i in tqdm.auto.tqdm(range(MAX_STEPS+1)):
    # 1. Sample Batches
    batch = train_data[torch.randint(0, len(train_data), (batch_size,), device=device)]
    c_idxs = torch.randint(0, len(train_data), (batch_size * control_mult,), device=device)
    control_batch = train_data[c_idxs]

    # 2. Compute Initial Gradients (The "Candidate" Gradients)
    # We detach these because we aren't optimizing the model parameters in the inner loop,
    # we are optimizing the *mask* that multiplies these fixed gradients.
    logits = model(batch["x"].to(device))
    loss = torch.nn.functional.cross_entropy(logits, batch["y"].to(device))
    
    # train_grads is a tuple of tensors
    train_grads = torch.autograd.grad(loss, model.parameters(), create_graph=False)
    train_grads = [g.detach() for g in train_grads] 
    orig_grad_norm = get_global_norm(train_grads)

    # 3. Initialize Mask
    # One scalar per parameter, initialized to 0.5 (neutral)
    # We require gradients for these masks.
    masks = [torch.full_like(p, 0.5, requires_grad=True, device=device) for p in model.parameters()]
    
    # Simple optimizer for the mask (SGD is usually sufficient for inner loops)
    mask_optim = torch.optim.SGD(masks, lr=mask_lr)

    # 4. Inner Loop: Optimize Mask
    for _ in range(mask_steps):
        mask_optim.zero_grad()
        
        # A. Create "Virtual" Weights: w' = w - lr * (grad * mask)
        # We simulate what the model WOULD look like if we applied this masked gradient.
        updated_params = {}
        for (name, param), grad, mask in zip(model.named_parameters(), train_grads, masks):
            # Note: We use simple SGD update rule here for the simulation
            updated_params[name] = param - model_lr * (grad * mask)
            
        # B. Compute Control Loss using Virtual Weights
        # functional_call allows us to run aux_model using the dictionary of weights 'updated_params'
        # This keeps the graph connected to 'masks'
        ctrl_logits = functional_call(aux_model, updated_params, (control_batch["x"].to(device),))
        ctrl_loss = torch.nn.functional.cross_entropy(ctrl_logits, control_batch["y"].to(device))
        
        # C. Add Regularization (L1 norm to push masks towards zero)
        # We want the minimal necessary mask to solve the control task
        reg_loss = 0
        for m in masks:
            reg_loss += torch.sum(torch.abs(m))
        
        total_loss = ctrl_loss + (mask_reg * reg_loss)
        
        # D. Optimize Mask
        total_loss.backward()
        mask_optim.step()
        
        # E. Clamp Mask to [0, 1]
        with torch.no_grad():
            for m in masks:
                m.clamp_(0.0, 1.0)

    # 5. Binarize Mask (Quantile Thresholding)
    all_masks_flat = flatten_tensors(masks)
    # We pick the threshold that keeps top N% of the optimized mask values
    threshold = torch.quantile(all_masks_flat, 1 - keep_top_p)
    
    final_grads = []
    for g, m in zip(train_grads, masks):
        # Hard binarization
        bin_mask = (m >= threshold).float()
        
        # Apply to the original gradients
        # Note: You can add rescaling here if desired (as in the previous iteration)
        final_grads.append(g * bin_mask)

    # Rescale final gradients to match original gradient norm
    final_grad_norm = get_global_norm(final_grads)
    scale_factor = orig_grad_norm / (final_grad_norm + 1e-8)
    final_grads = [g * scale_factor for g in final_grads]

    # 6. Apply Final Update to Real Model
    optim.zero_grad()
    for param, final_g in zip(model.parameters(), final_grads):
        param.grad = final_g
    optim.step()

    if i % LOG_EVERY == 0:
        acc = model(valid_data["x"]).argmax(dim=-1).eq(valid_data["y"]).float().mean().item()
        print(f"Iteration {i}: val acc = {acc:.4f}")